In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Week 11: Vector Databases & Semantic Search
## Part 1: Install Required Libraries

In [21]:
!pip install chromadb langchain-chroma openai python-dotenv

## Part 1: Generate Embeddings

In [22]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"
print("Key set:", os.getenv("OPENROUTER_API_KEY")[:10])

Key set: sk-or-v1-9


In [23]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [24]:
"text-embedding-ada-002"

'text-embedding-ada-002'

In [25]:
import os
from openai import OpenAI

os.environ["OPENROUTER_API_KEY"] = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return response.data[0].embedding


text = "vacation policy"
embedding = get_embedding(text)

print("Embedding length:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding length: 1536
First 5 values: [-0.012554051354527473, -0.009891919791698456, 0.014088279567658901, -0.027560066431760788, -0.02132507413625717]


Task 1.2: Calculate Similarity

In [26]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors

    Returns:
        Float between -1 and 1
    """

    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    dot_product = np.dot(vec1, vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)


# Test phrases
phrases = [
    "vacation policy",
    "time off rules",
    "PTO guidelines",
    "dress code requirements"
]

# Generate embeddings
embeddings = [get_embedding(p) for p in phrases]

# Base phrase embedding
base = embeddings[0]

print(f'Comparing "{phrases[0]}" with:\n')

# Compare similarities
for i, phrase in enumerate(phrases[1:], start=1):

    similarity = cosine_similarity(base, embeddings[i])

    print(f"{phrase:30} Similarity: {similarity:.4f}")

Comparing "vacation policy" with:

time off rules                 Similarity: 0.8467
PTO guidelines                 Similarity: 0.7709
dress code requirements        Similarity: 0.7585


## PART 2: CHROMADB SETUP
### Task 2.1: Initialize ChromaDB

In [27]:
import os
import chromadb
import requests
from chromadb.api.types import EmbeddingFunction

# -----------------------------
# OpenRouter Embedding Wrapper
# -----------------------------
class OpenRouterEmbeddingFunction(EmbeddingFunction):
    def __init__(self, api_key, model="openai/text-embedding-3-small"):
        self.api_key = api_key
        self.model = model

    def __call__(self, input):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers=headers,
            json={
                "model": self.model,
                "input": input
            }
        )

        if response.status_code != 200:
            raise Exception(f"Embedding API error: {response.text}")

        data = response.json()
        return [item["embedding"] for item in data["data"]]


# -----------------------------
# Setup
# -----------------------------
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found")

chroma_client = chromadb.PersistentClient(path="./chroma_db")

embedding_fn = OpenRouterEmbeddingFunction(
    api_key=api_key,
    model="openai/text-embedding-3-small"
)

collection = chroma_client.get_or_create_collection(
    name="company_docs",
    embedding_function=embedding_fn,
    metadata={"description": "Company policy documents"}
)

print("Collection Name:", collection.name)
print("Total Documents:", collection.count())

Collection Name: company_docs
Total Documents: 3


Task 2.2: Load and Index Documents

In [28]:
pip install -U langchain langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [29]:
import os

# Create folder
os.makedirs("company_docs", exist_ok=True)

print("✅ Folder 'company_docs' created successfully!")

✅ Folder 'company_docs' created successfully!


In [30]:
hr_content = """
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Leave:
12 weeks paid parental leave for primary caregivers.
6 weeks paid leave for secondary caregivers.
"""

with open("company_docs/hr_policy.txt", "w", encoding="utf-8") as f:
    f.write(hr_content)

print("✅ hr_policy.txt created")

✅ hr_policy.txt created


In [31]:
benefits_content = """
Employee Benefits

Health Insurance:
The company provides medical, dental, and vision insurance
for all full-time employees.

Retirement Plan:
Employees are eligible for the 401(k) retirement plan
after 6 months of employment.

Gym Membership:
Employees receive a free gym membership reimbursement
up to $50 per month.

Annual Bonus:
Performance-based annual bonuses are provided
at the end of each year.
"""

with open("company_docs/benefits.txt", "w", encoding="utf-8") as f:
    f.write(benefits_content)

print("✅ benefits.txt created")

✅ benefits.txt created


In [32]:
it_content = """
IT Security Policy

Password Rules:
Employees must use strong passwords with at least 8 characters.
Passwords should be changed every 90 days.

Internet Usage:
Company internet should only be used for work-related purposes.

Device Policy:
Employees must not install unauthorized software
on company laptops.

Data Security:
Confidential company data must not be shared externally
without permission.
"""

with open("company_docs/it_policy.txt", "w", encoding="utf-8") as f:
    f.write(it_content)

print("✅ it_policy.txt created")

✅ it_policy.txt created


In [33]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -----------------------------
# Load documents
# -----------------------------
loader = DirectoryLoader(
    "company_docs/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

# -----------------------------
# Split into chunks
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

# -----------------------------
# Check existing data
# -----------------------------
existing_count = collection.count()

if existing_count == 0:
    ids = []
    texts = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        ids.append(f"doc_{i}_{hash(chunk.page_content)}")  # safer unique ID
        texts.append(chunk.page_content)

        metadatas.append({
            "source": chunk.metadata.get("source", "unknown"),
            "chunk_index": i
        })

    # -----------------------------
    # Add to ChromaDB
    # -----------------------------
    collection.add(
        documents=texts,
        ids=ids,
        metadatas=metadatas
    )

    print(f"✅ Added {len(chunks)} chunks to ChromaDB")

else:
    print(f"⚠️ Collection already contains {existing_count} documents. Skipping ingestion.")

Loaded 3 documents
Total chunks created: 3
⚠️ Collection already contains 3 documents. Skipping ingestion.


# PART 3: SEMANTIC RAG

## Task 3.1: Test Vector Search

In [34]:
def vector_search(query, n_results=3):
    """
    Semantic search using ChromaDB
    """
    return collection.query(
        query_texts=[query],
        n_results=n_results
    )


def print_search_results(query, results):
    """
    Pretty-print search results safely
    """
    print("\n" + "=" * 70)
    print(f"🔎 Query: {query}")
    print("=" * 70)

    documents = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not documents:
        print("No results found.")
        return

    for i, doc in enumerate(documents):
        distance = distances[i] if i < len(distances) else None

        print(f"\nResult {i + 1}")
        if distance is not None:
            print(f"Similarity score (distance): {distance:.4f}")

        print("-" * 50)
        print(doc[:300] + "...")


# -----------------------------
# Test semantic understanding
# -----------------------------
test_queries = [
    "time off policy",   # should match vacation policy
    "WFH guidelines",    # should match remote work policy
    "maternity leave"    # should match parental leave
]

for query in test_queries:
    results = vector_search(query, n_results=2)
    print_search_results(query, results)


🔎 Query: time off policy

Result 1
Similarity score (distance): 0.9020
--------------------------------------------------
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Le...

Result 2
Similarity score (distance): 1.2378
--------------------------------------------------
IT Security Policy

Password Rules:
Employees must use strong passwords with at least 8 characters.
Passwords should be changed every 90 days.

Internet Usage:
Company internet should only be used for work-related purposes.

Device Policy:
Employees must not install unauthorized software
on company ...

🔎 Query: WFH guidelines

Result 1
Similarity score (distance): 1.5279
--------------------------------------------------
IT Security Policy

Password Rules:
Employees must

## Task 3.2: Build Semantic RAG Pipeline

In [36]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.6 MB/s eta 0:00:0000:01
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0


In [40]:
from langchain_openai import ChatOpenAI

In [41]:
from langchain_openai import ChatOpenAI


# -----------------------------
# LLM Setup (OpenRouter compatible)
# -----------------------------
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# -----------------------------
# RAG Pipeline
# -----------------------------
def semantic_rag(query, n_results=3):
    """
    Complete semantic RAG pipeline with ChromaDB + OpenRouter
    """

    # Step 1: Vector search
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "❌ No relevant information found in the knowledge base."

    # Step 2: Build context (safe + limited)
    context = "\n\n---\n\n".join(documents[:n_results])

    # Step 3: System + user prompt (better RAG behavior)
    messages = [
        {
            "role": "system",
            "content": (
                "You are an HR assistant. "
                "Answer ONLY using the provided context. "
                "If the answer is not in the context, say: "
                "'I don't have enough information in the documents.'"
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{query}

Answer clearly and concisely:
"""
        }
    ]

    # Step 4: Generate response
    response = llm.invoke(messages)

    return response.content


# -----------------------------
# Test Queries
# -----------------------------
test_questions = [
    "How much time off do employees get?",
    "Can I work from home?",
    "What is the maternity leave policy?"
]

for q in test_questions:
    print("\n" + "=" * 60)
    print(f"Q: {q}")
    print("=" * 60)
    print(semantic_rag(q))


Q: How much time off do employees get?
Employees receive 15 days of paid vacation per year.

Q: Can I work from home?
Employees may work remotely up to 3 days per week with manager approval.

Q: What is the maternity leave policy?
12 weeks paid parental leave for primary caregivers.
6 weeks paid leave for secondary caregivers.


# BONUS: KEYWORD VS SEMANTIC COMPARISON

In [42]:
from collections import Counter
import re


# -----------------------------
# Improved Keyword Search
# -----------------------------
def keyword_search(query, chunks, top_k=3):
    """
    Simple TF-style keyword scoring (improved version)
    """
    query_words = re.findall(r'\w+', query.lower())
    scored = []

    for chunk in chunks:
        text = chunk.page_content.lower()
        text_words = re.findall(r'\w+', text)

        text_counter = Counter(text_words)

        # score = sum of query word frequencies in chunk
        score = sum(text_counter[word] for word in query_words)

        if score > 0:
            scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [chunk for score, chunk in scored[:top_k]]


# -----------------------------
# Semantic search wrapper (safe)
# -----------------------------
def semantic_search(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    docs = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    return list(zip(docs, distances))


# -----------------------------
# Comparison function
# -----------------------------
def compare_search(query, chunks):
    print("\n" + "=" * 80)
    print(f"🔎 QUERY: {query}")
    print("=" * 80)

    # ---------------- Keyword ----------------
    print("\n🔤 KEYWORD SEARCH RESULTS:")
    kw_results = keyword_search(query, chunks, top_k=2)

    if not kw_results:
        print("No keyword matches found.")
    else:
        for i, chunk in enumerate(kw_results):
            print(f"\nResult {i+1}")
            print(chunk.page_content[:200] + "...")

    # ---------------- Semantic ----------------
    print("\n🧠 SEMANTIC SEARCH RESULTS:")
    sem_results = semantic_search(query, n_results=2)

    if not sem_results:
        print("No semantic matches found.")
    else:
        for i, (doc, dist) in enumerate(sem_results):
            print(f"\nResult {i+1}")
            print(f"Distance: {dist:.4f}")
            print(doc[:200] + "...")


# -----------------------------
# Run comparison
# -----------------------------
query = "PTO policy"  # should map to vacation policy

compare_search(query, chunks)


🔎 QUERY: PTO policy

🔤 KEYWORD SEARCH RESULTS:

Result 1
IT Security Policy

Password Rules:
Employees must use strong passwords with at least 8 characters.
Passwords should be changed every 90 days.

Internet Usage:
Company internet should only be used for...

Result 2
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Em...

🧠 SEMANTIC SEARCH RESULTS:

Result 1
Distance: 1.3401
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Em...

Result 2
Distance: 1.3964
IT Security Policy

Password Rules:
Employees must use strong passwords with at least 8 characters.
Passwords should be changed every 90 days.

Internet Usage:
Company internet should only be used for...
